# Benchmark dataloders of Tahoe100M dataset

In [1]:
from pathlib import Path

import json

In [2]:
VENV_PATH = Path("/home/sagemaker-user/venvs/annbatch/bin/python")
PATH_ZARR = Path("/home/sagemaker-user/data/tahoe100M/zarr_shuffled")
PATH_H5AD = Path("/home/sagemaker-user/data/tahoe100M/h5ad_shuffled")

## Arrayloaders

In [3]:
%%capture cap --no-stderr
!{VENV_PATH} arrayloader_benchmarks/annbatch/benchmark_annbatch.py --chunk_size=64 --preload_nchunks=256 --n_samples=2000000 --store_path={PATH_ZARR}

In [4]:
arrayloaders_64 = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
arrayloaders_64

23164.01712985015

In [5]:
%%capture cap --no-stderr
!{VENV_PATH} arrayloader_benchmarks/annbatch/benchmark_annbatch.py --chunk_size=128 --preload_nchunks=128 --n_samples=2000000 --store_path={PATH_ZARR}

In [6]:
arrayloaders_128 = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
arrayloaders_128

29758.370564857705

In [7]:
%%capture cap --no-stderr
!{VENV_PATH} arrayloader_benchmarks/annbatch/benchmark_annbatch.py --chunk_size=256 --preload_nchunks=64 --n_samples=2000000 --store_path={PATH_ZARR}

In [8]:
arrayloaders_256 = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
arrayloaders_256

34265.97570459475

In [28]:
%%capture cap --no-stderr
!{VENV_PATH} arrayloader_benchmarks/annbatch/benchmark_annbatch.py --chunk_size=512 --preload_nchunks=32 --n_samples=2000000 --store_path={PATH_ZARR}

In [29]:
arrayloaders_512 = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
arrayloaders_512

35007.23403382923

In [14]:
%%capture cap --no-stderr
!{VENV_PATH} arrayloader_benchmarks/annbatch/benchmark_annbatch.py --chunk_size=512 --preload_nchunks=32 --preload_to_gpu=True --n_samples=2000000 --store_path={PATH_ZARR}

In [15]:
arrayloaders_gpu = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
arrayloaders_gpu

35812.0438494987

## MappedCollection

In [16]:
%%capture cap --no-stderr
!{VENV_PATH} arrayloader_benchmarks/mapped_collection/benchmark_mapped_collection.py --store_path={PATH_H5AD}

In [17]:
mapped_collection = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
mapped_collection

852.5874964847992

## scDataset

In [21]:
%%capture cap --no-stderr
# use only 6 workers to not run out of memory
!{VENV_PATH} arrayloader_benchmarks/scDataset/benchmark_scDataset.py --num_workers=6 --batch_size=64 --store_path={PATH_H5AD}  # recommended settings from tutorial

In [22]:
sc_dataset = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
sc_dataset

1494.8052171850566

In [23]:
%%capture cap --no-stderr
# use only 6 workers to not run out of memory
!{VENV_PATH} arrayloader_benchmarks/scDataset/benchmark_scDataset.py --num_workers=6 --batch_size=4096 --block_size=256 --fetch_factor=4 --store_path={PATH_H5AD}

In [24]:
sc_dataset_matched = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
sc_dataset_matched

17933.122392184283

## Plot results

In [30]:
import pandas as pd

results = pd.DataFrame(
    {
        "loader": [
            "annbatch (chunk_size=64)",
            "annbatch (chunk_size=128)",
            "annbatch (chunk_size=256)",
            "annbatch (chunk_size=512)",
            "annbatch (GPU)",
            "MappedCollection",
            "scDataset (recommended settings)",
            "scDataset (annbatch matched settings)",
        ],
        "samples/sec": [
            arrayloaders_64,
            arrayloaders_128,
            arrayloaders_256,
            arrayloaders_512,
            arrayloaders_gpu,
            mapped_collection,
            sc_dataset,
            sc_dataset_matched,
        ],
    }
)
results.to_csv("loading_speed_results_AWS.csv", index=False)
results

,loader,samples/sec
0,annbatch (chunk_size=64),23164.017130
1,annbatch (chunk_size=128),29758.370565
2,annbatch (chunk_size=256),34265.975705
3,annbatch (chunk_size=512),35007.234034
4,annbatch (GPU),35812.043849
5,MappedCollection,852.587496
6,scDataset (recommended settings),1494.805217
7,scDataset (annbatch matched settings),17933.122392
